In [2]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ================= 配置区 =================
RAW_KG_PATH = "kg.csv"

def batch_inspect_ontology():
    print("==================================================")
    print("🛠️ 步骤 1: 启动 PrimeKG 本体对齐批量检修工具...")
    
    try:
        # low_memory=False 防止大文件读取警告
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
        print(f"   ✅ 成功加载 {RAW_KG_PATH}。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {RAW_KG_PATH}，请检查路径。")
        return

    print("\n==================================================")
    print("🧹 步骤 2: 划定安全检索范围...")
    # 过滤出只属于疾病、表型和基因/蛋白的子集
    target_types = ['disease', 'effect/phenotype', 'gene/protein']
    search_df = df[df['x_type'].isin(target_types)]
    print(f"   已将检索范围锁定在 {len(search_df)} 条 'disease' 和 'effect/phenotype' 关系中。")
    
    # 针对AIBL特征和新增标志物定义搜索策略
    search_strategy = {
        "mmse / cdr (认知/总体损伤)": ["cognition", "cognitive impairment", "dementia"],
        "lm_imm / lm_del (记忆力)": ["memory"],
        "apoe (载脂蛋白E)": ["apoe", "apolipoprotein E"],
        "GFAP (胶质纤维酸性蛋白)": ["gfap", "glial fibrillary acidic protein"],
        "NfL (神经丝轻链)": ["nefl", "neurofilament light"],
        "Tau (微管相关蛋白)": ["mapt", "tau"]
    }

    print("\n==================================================")
    print("🔍 步骤 3: 开始按医学词根进行批量模糊检索...")
    
    for category, keywords in search_strategy.items():
        print(f"\n▶ 正在检索【{category}】...")
        print(f"   使用的词根: {keywords}")
        
        # 构建正则表达式，多个词根用 | 连接
        pattern = '|'.join(keywords)
        
        # 在 x_name 列进行正则匹配，case=False 表示忽略大小写
        mask = search_df['x_name'].str.contains(pattern, case=False, na=False)
        results = search_df[mask]['x_name'].unique()
        
        if len(results) > 0:
            print(f"   ✅ 找到 {len(results)} 个候选节点，前 10 个最精准的候选如下：")
            # 简单排序：把长度较短的（通常是更基础的本体标准词）放前面
            sorted_results = sorted(results, key=len)
            for i, res in enumerate(sorted_results[:10]):
                print(f"      [{i+1}] {res}")
        else:
            print(f"   ❌ 使用这些词根未能找到任何节点，PrimeKG 中可能使用了完全不同的医学术语。")

    print("\n==================================================")
    print("💡 步骤 4: 检修结果输出完毕。")
    print("   请从上述候选项中挑选最贴切的准确学名（请原样复制，区分大小写），")
    print("   将它们替换回第一个脚本的 aibl_mapping 字典中。")
    print("==================================================")

if __name__ == "__main__":
    batch_inspect_ontology()

🛠️ 步骤 1: 启动 PrimeKG 本体对齐批量检修工具...
   ✅ 成功加载 kg.csv。

🧹 步骤 2: 划定安全检索范围...
   已将检索范围锁定在 3229569 条 'disease' 和 'effect/phenotype' 关系中。

🔍 步骤 3: 开始按医学词根进行批量模糊检索...

▶ 正在检索【mmse / cdr (认知/总体损伤)】...
   使用的词根: ['cognition', 'cognitive impairment', 'dementia']
   ✅ 找到 45 个候选节点，前 10 个最精准的候选如下：
      [1] Dementia
      [2] genetic dementia
      [3] semantic dementia
      [4] vascular dementia
      [5] Semantic dementia
      [6] Lewy body dementia
      [7] dementia (disease)
      [8] Cognitive impairment
      [9] ataxia with dementia
      [10] Subcortical dementia

▶ 正在检索【lm_imm / lm_del (记忆力)】...
   使用的词根: ['memory']
   ✅ 找到 45 个候选节点，前 10 个最精准的候选如下：
      [1] Memory impairment
      [2] Procedural memory loss
      [3] Declarative memory loss
      [4] Absence of memory B cells
      [5] Long term memory impairment
      [6] Short term memory impairment
      [7] Retrograde memory impairment
      [8] Anterograde memory impairment
      [9] Impaired memory B cell generation
      [10] Ab